In [72]:
#| default_exp dmelements

# dmelements
> Operators on density-matrix elements

## Imports -

In [73]:
#| hide
from fastcore.test import test_close

In [74]:
#| export
import importlib

from pylgs.imports import *
from pylgs.utilities.nbdev import DictTbl, AttributeTbl
from pylgs.utilities.testing import test_array
from pylgs.utilities.sparse import sparse_kronecker_matrix, sparse_toeplitz, sparse_identity, sparse_diag, sparse
from pylgs.utilities.numpy import sym_range
from pylgs.pymor.parameters import *
from pylgs.pymor.vectorarrays import *
from pylgs.pymor.operators import *
from pylgs.pymor.timestepping import *
from pylgs.pymor.grids import *
from pylgs.pymor.models import *
from pymor.vectorarrays.interface import VectorArray
from pymor.operators.interface import Operator

In [75]:
#| hide
np.set_printoptions(formatter={'float': lambda x: f'{x:^ 8.2}' if x else f'{0:^ 8}'}, linewidth=140)

## DM elements -

In [76]:
#| export
_dm_element_hfz = re.compile(r'ρ<sub>([^,]+), \(([^,]+), ([^,]+), ([^)]+)\), \(([^,]+), ([^,]+), ([^)]+)\)</sub>')
_dm_element_toy = re.compile(r'ρ<sub>([^,]+), ([^,]+), ([^,]+)</sub>')

In [77]:
#| hide
with importlib.resources.path("pylgs.systems.NaD1_Toy", "Flux.mtxn") as path:
    flux = LincombOperator.from_file(path)
dm_elements = flux.assemble().matrix['Density matrix' + Lbl.SRC].rename({"Density matrix (source)": "Density matrix"})
dm_elements

<xarray.DataArray 'Density matrix (source)' (Density matrix: 4)> Size: 800B
array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>',
       'ρ<sub>Re, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
       'ρ<sub>Im, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
       'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50')
Coordinates:
  * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...

In [78]:
#| hide
with importlib.resources.path("pylgs.systems.NaD1", "Flux.mtxn") as path:
    flux = LincombOperator.from_file(path)
dm_elements_hfz = flux.assemble().matrix['Density matrix' + Lbl.SRC].rename({"Density matrix (source)": "Density matrix"})
dm_elements_hfz[:10]

<xarray.DataArray 'Density matrix (source)' (Density matrix: 10)> Size: 3kB
array(['ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 2)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
       'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 1)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 0)</sub>',
       'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 0)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 0)</sub>',
       'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 0)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, 0)</sub>',
       'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, -1)</sub>'],
      dtype='<U68')
Coordinates:
  * Density matrix  (Density matrix) <U68 3kB 'ρ<sub>Re, (3S<sub>1/2</sub>, 2...

### parse_dm_element -


In [79]:
#| export
def parse_dm_element(s):
    if match := _dm_element_hfz.findall(s):
        return match[0][0], match[0][1:4], match[0][4:7] 
    if match := _dm_element_toy.findall(s): 
        return match[0][0], (match[0][1],), (match[0][2],)

In [80]:
#| hide
[parse_dm_element(e) for e in dm_elements.data]

[('Re', ('3S<sub>1/2</sub>',), ('3S<sub>1/2</sub>',)),
 ('Re', ('3S<sub>1/2</sub>',), ('3P<sub>1/2</sub>',)),
 ('Im', ('3S<sub>1/2</sub>',), ('3P<sub>1/2</sub>',)),
 ('Re', ('3P<sub>1/2</sub>',), ('3P<sub>1/2</sub>',))]

In [81]:
#| hide
[parse_dm_element(e) for e in dm_elements_hfz.data][::10]

[('Re', ('3S<sub>1/2</sub>', '2', '2'), ('3S<sub>1/2</sub>', '2', '2')),
 ('Im', ('3S<sub>1/2</sub>', '2', '2'), ('3S<sub>1/2</sub>', '2', '-1')),
 ('Re', ('3S<sub>1/2</sub>', '2', '0'), ('3S<sub>1/2</sub>', '2', '-2')),
 ('Im', ('3S<sub>1/2</sub>', '1', '1'), ('3S<sub>1/2</sub>', '1', '-1')),
 ('Re', ('3S<sub>1/2</sub>', '2', '-1'), ('3P<sub>1/2</sub>', '2', '2')),
 ('Re', ('3P<sub>1/2</sub>', '2', '2'), ('3P<sub>1/2</sub>', '2', '2')),
 ('Im', ('3S<sub>1/2</sub>', '2', '-2'), ('3P<sub>1/2</sub>', '2', '1')),
 ('Re', ('3S<sub>1/2</sub>', '2', '2'), ('3P<sub>1/2</sub>', '2', '0')),
 ('Re', ('3S<sub>1/2</sub>', '1', '1'), ('3P<sub>1/2</sub>', '2', '0')),
 ('Re', ('3P<sub>1/2</sub>', '2', '0'), ('3P<sub>1/2</sub>', '2', '0')),
 ('Im', ('3S<sub>1/2</sub>', '2', '-2'), ('3P<sub>1/2</sub>', '2', '-1')),
 ('Im', ('3P<sub>1/2</sub>', '2', '1'), ('3P<sub>1/2</sub>', '2', '-1')),
 ('Re', ('3S<sub>1/2</sub>', '2', '-1'), ('3P<sub>1/2</sub>', '2', '-2')),
 ('Re', ('3P<sub>1/2</sub>', '2', '2'), (

### population_element -


In [82]:
#| export
@dispatch
def population_element(s):
    parsed_dm = parse_dm_element(s)
    return parsed_dm[1] == parsed_dm[2]

@dispatch
def population_element(expr:DataArray):
    return xr.apply_ufunc(population_element, expr, vectorize=True)

In [83]:
#| hide
[population_element(e) for e in dm_elements.data]

[True, False, False, True]

In [84]:
#| hide
[population_element(e) for e in dm_elements_hfz.data][::10]

[True,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False]

In [85]:
population_element(dm_elements_hfz)

<xarray.DataArray 'Density matrix (source)' (Density matrix: 196)> Size: 196B
array([ True, False, False,  True, False, False, False, False,  True, False, False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False,  True,  True, False, False,  True, False, False, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False,  True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True])
Coordinates:
  * Density matrix  (Density matrix) <U68 53kB 'ρ<sub>Re, (3S<sub>1/2</sub>, ...

### level_label -


In [86]:
#| export
@dispatch
def level_label(expr:str):
    parsed = parse_dm_element(expr)
    if parsed[1][0] != parsed[2][0]: return ''
    return parsed[1][0]

@dispatch
def level_label(expr:DataArray):
    return xr.apply_ufunc(level_label, expr, vectorize=True)

In [87]:
#| hide
level_label(dm_elements[0])

<xarray.DataArray 'Density matrix (source)' ()> Size: 64B
array('3S<sub>1/2</sub>', dtype='<U16')
Coordinates:
    Density matrix  <U50 200B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></...

In [88]:
#| hide
level_label(dm_elements)

<xarray.DataArray 'Density matrix (source)' (Density matrix: 4)> Size: 256B
array(['3S<sub>1/2</sub>', '', '', '3P<sub>1/2</sub>'], dtype='<U16')
Coordinates:
  * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...

In [89]:
#| hide
level_label(dm_elements_hfz)

<xarray.DataArray 'Density matrix (source)' (Density matrix: 196)> Size: 13kB
array(['3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>',
       '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>',
       '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>',
       '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>',
       '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>',
       '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '3S<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '',
       '', '', '', '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '3P<sub>1/2</sub>',
       '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '3P<sub>1/2</sub>',
       '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '',
       '', '', '', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>',
       '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>',
       '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>',
       '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '3P<sub>1/2</sub>', '', '', '', '', '', '', '',
       '', '', '', '', '', '', '', '', '', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '', '', '', '', '', '', '', '', '',
       '', '', '', '', '', '', '', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>', '3P<sub>1/2</sub>'],
      dtype='<U16')
Coordinates:
  * Density matrix  (Density matrix) <U68 53kB 'ρ<sub>Re, (3S<sub>1/2</sub>, ...

### levels -

In [90]:
#| export
def levels(dm_elements):
    return xr.Variable(
        'Level', 
        list(set().union(filter(None, xr.apply_ufunc(level_label, dm_elements, vectorize=True).data)))
    )

In [91]:
#| hide
levels(dm_elements)

<xarray.Variable (Level: 2)> Size: 128B
array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16')

In [92]:
#| hide
levels(dm_elements_hfz)

<xarray.Variable (Level: 2)> Size: 128B
array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16')

### population -

In [93]:
#| export
def population(dm_elements):
    elements = dm_elements.data
    mask = [population_element(e) for e in elements]
    pop_labels = [level_label(e) for e in elements[mask]]
    return XarrayMatrixOperator(
        DataArray(
            np.identity(len(elements))[mask],
            coords=[
                ("Population (range)", elements[mask]),
                ("Density matrix (source)", elements), 
            ],
        )
    )

In [94]:
op = population(dm_elements)
op

XarrayMatrixOperator(
    <xarray.DataArray (Population (range): 2, Density matrix (source): 4)> Size: 64B
    array([[   1.0  ,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   ,    1.0  ]])
    Coordinates:
      * Population (range)       (Population (range)) <U50 400B 'ρ<sub>Re, 3S<sub...
      * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...,
    source=XarrayVectorSpace(
               coord_data={Density matrix: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
                                  'ρ<sub>Im, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}),
    range=XarrayVectorSpace(
              coord_data={Population: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}))

In [95]:
dm = op.source.ones()

In [96]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(
        coord_data={Population: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}),
    XarrayVectorArrayImpl(
        array([   1.0  ,    1.0  ]),
        XarrayVectorSpace(
            coord_data={Population: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}),
        _data_dims=['Population'],
        _extended_coord_data={}),
    _len=1)

In [97]:
pop.array

<xarray.DataArray (Population: 2)> Size: 16B
array([   1.0  ,    1.0  ])
Coordinates:
  * Population  (Population) <U50 400B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/...

In [98]:
op = population(dm_elements_hfz)
op

XarrayMatrixOperator(
    <xarray.DataArray (Population (range): 16, Density matrix (source): 196)> Size: 25kB
    array([[   1.0  ,     0   ,     0   , ...,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   , ...,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   , ...,     0   ,     0   ,     0   ],
           ...,
           [    0   ,     0   ,     0   , ...,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   , ...,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   , ...,     0   ,     0   ,    1.0  ]], shape=(16, 196))
    Coordinates:
      * Population (range)       (Population (range)) <U68 4kB 'ρ<sub>Re, (3S<sub...
      * Density matrix (source)  (Density matrix (source)) <U68 53kB 'ρ<sub>Re, (...,
    source=XarrayVectorSpace(
               coord_data={Density matrix: (array(['ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -1), (3S<sub>1/2</sub>, 2, -1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -1), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, -1), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -2), (3S<sub>1/2</sub>, 2, -2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, 1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, 0)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, 0)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 0), (3S<sub>1/2</sub>, 1, 0)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, -1)</sub>',
           

In [99]:
dm = op.source.ones()

In [100]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(
        coord_data={Population: (array(['ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 2)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 1)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, 0)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -1), (3S<sub>1/2</sub>, 2, -1)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -2), (3S<sub>1/2</sub>, 2, -2)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, 1)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 0), (3S<sub>1/2</sub>, 1, 0)</sub>',
                           'ρ<sub>Re, (3S<sub>1/2</sub>, 1, -1), (3S<sub>1/2</sub>, 1, -1)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 2), (3P<sub>1/2</sub>, 2, 2)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 1), (3P<sub>1/2</sub>, 2, 1)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 0), (3P<sub>1/2</sub>, 2, 0)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 2, -1), (3P<sub>1/2</sub>, 2, -1)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 2, -2), (3P<sub>1/2</sub>, 2, -2)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 1, 1), (3P<sub>1/2</sub>, 1, 1)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 1, 0), (3P<sub>1/2</sub>, 1, 0)</sub>',
                           'ρ<sub>Re, (3P<sub>1/2</sub>, 1, -1), (3P<sub>1/2</sub>, 1, -1)</sub>'], dtype='<U68'), {})}),
    XarrayVectorArrayImpl(
        array([   1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,
                  1.0  ,    1.0  ,    1.0  ]),
        XarrayVectorSpace(
            coord_data={Population: (array(['ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 2)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 1)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 0), (3S<sub>1/2</sub>, 2, 0)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -1), (3S<sub>1/2</sub>, 2, -1)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 2, -2), (3S<sub>1/2</sub>, 2, -2)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 1), (3S<sub>1/2</sub>, 1, 1)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 1, 0), (3S<sub>1/2</sub>, 1, 0)</sub>',
                               'ρ<sub>Re, (3S<sub>1/2</sub>, 1, -1), (3S<sub>1/2</sub>, 1, -1)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 2), (3P<sub>1/2</sub>, 2, 2)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 1), (3P<sub>1/2</sub>, 2, 1)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 2, 0), (3P<sub>1/2</sub>, 2, 0)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 2, -1), (3P<sub>1/2</sub>, 2, -1)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 2, -2), (3P<sub>1/2</sub>, 2, -2)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 1, 1), (3P<sub>1/2</sub>, 1, 1)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 1, 0), (3P<sub>1/2</sub>, 1, 0)</sub>',
                               'ρ<sub>Re, (3P<sub>1/2</sub>, 1, -1), (3P<sub>1/2</sub>, 1, -1)</sub>'], dtype='<U68'), {})}),
        _data_dims=['Population'],
        _extended_coord_data={}),
    _len=1)

In [101]:
pop.array

<xarray.DataArray (Population: 16)> Size: 128B
array([   1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,    1.0  ,
          1.0  ,    1.0  ,    1.0  ])
Coordinates:
  * Population  (Population) <U68 4kB 'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3...

### total_population - 

In [102]:
#| export
def total_population(dm_elements):
    return XarrayMatrixOperator(
        population(
            dm_elements
        ).matrix.sum(
            "Population (range)"
        ),
        name="Total population"
    )

In [103]:
op = total_population(dm_elements)
op

XarrayMatrixOperator(
    <xarray.DataArray (Density matrix (source): 4)> Size: 32B
    array([   1.0  ,     0   ,     0   ,    1.0  ])
    Coordinates:
      * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...,
    source=XarrayVectorSpace(
               coord_data={Density matrix: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
                                  'ρ<sub>Im, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}),
    range=XarrayVectorSpace(coord_data={}),
    name='Total population')

In [104]:
op.source

XarrayVectorSpace(
    coord_data={Density matrix: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
                       'ρ<sub>Im, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})})

In [105]:
op.range

XarrayVectorSpace(coord_data={})

In [106]:
dm = op.source.ones()

In [107]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(coord_data={}),
    XarrayVectorArrayImpl(array(   2.0  ), XarrayVectorSpace(coord_data={}), _data_dims=[], _extended_coord_data={}),
    _len=1)

In [108]:
pop.array

<xarray.DataArray ()> Size: 8B
array(   2.0  )

### level_population -


In [109]:
#|export
def level_mask_da(dm_elements):
    return xr.concat([level == level_label(dm_elements) for level in levels(dm_elements)], levels(dm_elements))

In [110]:
level_mask_da(dm_elements)

<xarray.DataArray 'Density matrix (source)' (Level: 2, Density matrix: 4)> Size: 8B
array([[False, False, False,  True],
       [ True, False, False, False]])
Coordinates:
  * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...
  * Level           (Level) <U16 128B '3P<sub>1/2</sub>' '3S<sub>1/2</sub>'

In [111]:
#|export
def level_population_mask_da(dm_elements):
    return level_mask_da(dm_elements) * population_element(dm_elements)

In [112]:
level_population_mask_da(dm_elements)

<xarray.DataArray 'Density matrix (source)' (Level: 2, Density matrix: 4)> Size: 8B
array([[False, False, False,  True],
       [ True, False, False, False]])
Coordinates:
  * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...
  * Level           (Level) <U16 128B '3P<sub>1/2</sub>' '3S<sub>1/2</sub>'

In [113]:
#|export
def level_population(dm_elements):
    return XarrayMatrixOperator(
            level_population_mask_da(dm_elements),
            source="Density matrix",
            range="Level",
            name="Population"
        )

In [114]:
op = level_population(dm_elements)
op

XarrayMatrixOperator(
    <xarray.DataArray 'Density matrix (source)' (Level (range): 2,
                                                 Density matrix (source): 4)> Size: 8B
    array([[False, False, False,  True],
           [ True, False, False, False]])
    Coordinates:
      * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...
      * Level (range)            (Level (range)) <U16 128B '3P<sub>1/2</sub>' '3S...,
    source=XarrayVectorSpace(
               coord_data={Density matrix: (array(['ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>', 'ρ<sub>Re, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>',
                                  'ρ<sub>Im, 3S<sub>1/2</sub>, 3P<sub>1/2</sub></sub>', 'ρ<sub>Re, 3P<sub>1/2</sub>, 3P<sub>1/2</sub></sub>'], dtype='<U50'), {})}),
    range=XarrayVectorSpace(coord_data={Level: (array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16'), {})}),
    name='Population')

In [115]:
op.matrix

<xarray.DataArray 'Density matrix (source)' (Level (range): 2,
                                             Density matrix (source): 4)> Size: 8B
array([[False, False, False,  True],
       [ True, False, False, False]])
Coordinates:
  * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...
  * Level (range)            (Level (range)) <U16 128B '3P<sub>1/2</sub>' '3S...

In [116]:
dm = op.source.ones()

In [117]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(coord_data={Level: (array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16'), {})}),
    XarrayVectorArrayImpl(
        array([   1.0  ,    1.0  ]),
        XarrayVectorSpace(coord_data={Level: (array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16'), {})}),
        _data_dims=['Level'],
        _extended_coord_data={}),
    _len=1)

In [118]:
pop.array

<xarray.DataArray (Level: 2)> Size: 16B
array([   1.0  ,    1.0  ])
Coordinates:
  * Level    (Level) <U16 128B '3P<sub>1/2</sub>' '3S<sub>1/2</sub>'

In [119]:
op = level_population(dm_elements_hfz)
op

XarrayMatrixOperator(
    <xarray.DataArray 'Density matrix (source)' (Level (range): 2,
                                                 Density matrix (source): 196)> Size: 392B
    array([[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
             True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False,  True, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False,  True, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True],
           [ True, False, False,  True, False, False, False, False,  True, False, False, False, False, False, False,  True, False, False,
            False, False, False, False, False, False,  True,  True, False, False,  True, False, False, False, False,  True, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]])
    Coordinates:
      * Density matrix (source)  (Density matrix (source)) <U68 53kB 'ρ<sub>Re, (...
      * Level (range)            (Level (range)) <U16 128B '3P<sub>1/2</sub>' '3S...,
    source=XarrayVectorSpace(
               coord_data={Density matrix: (array(['ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 2)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Im, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 1), (3S<sub>1/2</sub>, 2, 1)</sub>',
                                  'ρ<sub>Re, (3S<sub>1/2</sub>, 2, 2), (3S<sub>1/2</sub>, 2, 0)</sub>',
                                  'ρ<sub>Im, (3S<sub>1

In [120]:
dm = op.source.ones()

In [121]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(coord_data={Level: (array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16'), {})}),
    XarrayVectorArrayImpl(
        array([   8.0  ,    8.0  ]),
        XarrayVectorSpace(coord_data={Level: (array(['3P<sub>1/2</sub>', '3S<sub>1/2</sub>'], dtype='<U16'), {})}),
        _data_dims=['Level'],
        _extended_coord_data={}),
    _len=1)

In [122]:
pop.array

<xarray.DataArray (Level: 2)> Size: 16B
array([   8.0  ,    8.0  ])
Coordinates:
  * Level    (Level) <U16 128B '3P<sub>1/2</sub>' '3S<sub>1/2</sub>'

### sum_over_dm -


In [123]:
#| export
def sum_over_dm(dm_elements):
    return SumOperator({'Density matrix' + Lbl.SRC: dm_elements})

## Export -

In [124]:
#| hide
import nbdev; nbdev.nbdev_export()